# 🎙️ Arabic Text-to-Speech Training Pipeline

**Full end-to-end training notebook for Google Colab with persistent Google Drive storage.**

### Features
- ✅ Automatic dependency installation with Colab conflict resolution
- ✅ HuggingFace dataset import with automatic format conversion (mp3/flac → wav)
- ✅ Full Google Drive persistence (checkpoints, preprocessed data, phonemes, logs)
- ✅ Automatic training resume after disconnection
- ✅ Smart phase detection — never re-runs completed stages

---

## ⚙️ Cell 1 — Configuration

Edit these values **once** before your first run. All subsequent cells are fully automatic.

In [ ]:
#@title ⚙️ Configuration {display-mode: "form"}

#@markdown ### 📂 Google Drive Paths
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/tts_project"  #@param {type:"string"}

#@markdown ### 🗃️ HuggingFace Dataset
HF_DATASET_ID = "Mohamad-I8/AseelArabicDataset"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

#@markdown ### 🔧 GitHub Repository
GITHUB_REPO_URL = "https://github.com/shamsorachdi62/repo.git"  #@param {type:"string"}

#@markdown ### 🏋️ Training Parameters
EXPERIMENT_NAME = "saeed"  #@param {type:"string"}
BATCH_SIZE = 8  #@param {type:"integer"}
MAX_STEPS = 300000  #@param {type:"integer"}
SAMPLE_RATE = 24000  #@param {type:"integer"}
PREPROCESS_WORKERS = 2  #@param {type:"integer"}

# ────────────────────────────────────────────────────
# Derived paths — DO NOT edit below
# ────────────────────────────────────────────────────
import os

DRIVE_CHECKPOINTS   = os.path.join(DRIVE_PROJECT_ROOT, "checkpoints")
DRIVE_LOGS          = os.path.join(DRIVE_PROJECT_ROOT, "logs")
DRIVE_PREPROCESSED  = os.path.join(DRIVE_PROJECT_ROOT, "preprocessed_data")
DRIVE_RAW_DATASET   = os.path.join(DRIVE_PROJECT_ROOT, "raw_dataset")
DRIVE_CONVERTED_WAV = os.path.join(DRIVE_PROJECT_ROOT, "raw_dataset", "wav")
DRIVE_METADATA      = os.path.join(DRIVE_PROJECT_ROOT, "raw_dataset")
DRIVE_DATA_STATS    = os.path.join(DRIVE_PROJECT_ROOT, "data_stats")

LOCAL_REPO          = "/content/repo"
LOCAL_DATA_DIR      = os.path.join(LOCAL_REPO, "data", EXPERIMENT_NAME)

# Phase marker files on Drive
MARKER_DIR            = os.path.join(DRIVE_PROJECT_ROOT, ".markers")
MARKER_DOWNLOAD_DONE  = os.path.join(MARKER_DIR, "01_download_done")
MARKER_CONVERT_DONE   = os.path.join(MARKER_DIR, "02_convert_done")
MARKER_METADATA_DONE  = os.path.join(MARKER_DIR, "03_metadata_done")
MARKER_PREPROCESS_DONE= os.path.join(MARKER_DIR, "04_preprocess_done")
MARKER_STATS_DONE     = os.path.join(MARKER_DIR, "05_stats_done")

print("✅ Configuration loaded.")

## 📌 Cell 2 — Mount Google Drive & Create Directory Structure

In [ ]:
#@title 📌 Mount Google Drive {display-mode: "form"}

from google.colab import drive
import os, shutil

drive.mount('/content/drive', force_remount=False)

for d in [
    DRIVE_PROJECT_ROOT, DRIVE_CHECKPOINTS, DRIVE_LOGS,
    DRIVE_PREPROCESSED, DRIVE_RAW_DATASET, DRIVE_CONVERTED_WAV,
    DRIVE_DATA_STATS, MARKER_DIR
]:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive mounted and directory structure created.")
print(f"   Project root: {DRIVE_PROJECT_ROOT}")

## 📦 Cell 3 — Clone Repository & Install Dependencies

Handles Colab's pre-installed packages carefully to avoid version conflicts.

In [ ]:
#@title 📦 Clone Repo & Install Dependencies {display-mode: "form"}

import subprocess, sys, os, shutil

# ── Clone or update repository ──
if os.path.isdir(LOCAL_REPO):
    print("📂 Repository already exists locally. Pulling latest changes...")
    subprocess.run(["git", "pull"], cwd=LOCAL_REPO, check=False)
else:
    print("📥 Cloning repository...")
    subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, LOCAL_REPO],
        check=True
    )
    print("✅ Repository cloned.")

# ── Dependency installation strategy ──
# Colab pre-installs specific versions of torch, numpy, etc.
# We install the model's dependencies WITHOUT touching torch/numpy/torchaudio
# to avoid breaking Colab's CUDA runtime.

# Core dependencies that are safe to install/upgrade on Colab
SAFE_DEPS = [
    "lightning>=2.0.0",
    "torchmetrics>=0.11.4",
    "nnaudio>=0.3.3",
    "pyworld>=0.3.4",
    "hydra-core>=1.3.2",
    "hydra-colorlog>=1.2.0",
    "rootutils>=1.0.7",
    "rich>=13.7.1",
    "librosa==0.9.2",
    "einops>=0.8.0",
    "unidecode>=1.3.8",
    "onnx>=1.16.2",
    "onnxruntime>=1.18.1",
    "soundfile>=0.12.0",
    "datasets",
    "huggingface_hub",
    "pydub",
]

print("\n📦 Installing dependencies (preserving Colab's torch/numpy)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
     "lightning>=2.0.0", "torchmetrics>=0.11.4"],
    check=False
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + SAFE_DEPS,
    check=True
)

# Install ffmpeg for audio conversion
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

# Verify critical imports
print("\n🔍 Verifying critical imports...")
try:
    import torch
    print(f"   torch:      {torch.__version__} | CUDA: {torch.cuda.is_available()}")
    import torchaudio
    print(f"   torchaudio:  {torchaudio.__version__}")
    import lightning
    print(f"   lightning:   {lightning.__version__}")
    import numpy as np
    print(f"   numpy:       {np.__version__}")
    import librosa
    print(f"   librosa:     {librosa.__version__}")
    import hydra
    print(f"   hydra:       {hydra.__version__}")
    import onnxruntime
    print(f"   onnxruntime: {onnxruntime.__version__}")
    print("\n✅ All critical dependencies verified.")
except ImportError as e:
    print(f"\n❌ Missing dependency: {e}")
    print("   Try restarting the runtime and re-running this cell.")
    raise

# Add repo to Python path
if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

print(f"\n📁 Working directory: {os.getcwd()}")

## 🗃️ Cell 4 — Download & Convert HuggingFace Dataset

Downloads the dataset from HuggingFace, handles all archive/audio formats, and converts everything to **mono WAV at the target sample rate**. Results are cached on Google Drive.

In [ ]:
#@title 🗃️ Download & Convert Dataset {display-mode: "form"}

import os, glob, json, shutil, subprocess, traceback
from pathlib import Path

def touch_marker(path):
    Path(path).touch()

def marker_exists(path):
    return os.path.exists(path)

# ══════════════════════════════════════════════════════════
# PHASE 1: Download dataset from HuggingFace
# ══════════════════════════════════════════════════════════
if marker_exists(MARKER_DOWNLOAD_DONE):
    print("⏭️  Phase 1 (Download): Already completed — skipping.")
else:
    print("📥 Phase 1: Downloading dataset from HuggingFace...")
    try:
        from datasets import load_dataset

        load_kwargs = {"trust_remote_code": True}
        if HF_TOKEN:
            load_kwargs["token"] = HF_TOKEN

        # Try loading with the datasets library first
        try:
            ds = load_dataset(HF_DATASET_ID, **load_kwargs)
            print(f"   Dataset loaded via datasets library: {ds}")
            # The dataset is now in memory, we'll extract it in Phase 2
            _ds_loaded_via_lib = True
        except Exception as e_ds:
            print(f"   datasets library load failed: {e_ds}")
            print("   Falling back to huggingface_hub snapshot download...")
            from huggingface_hub import snapshot_download
            snapshot_download(
                repo_id=HF_DATASET_ID,
                repo_type="dataset",
                local_dir=os.path.join(DRIVE_RAW_DATASET, "_hf_snapshot"),
                token=HF_TOKEN or None,
            )
            _ds_loaded_via_lib = False
            print("   ✅ Snapshot downloaded.")

        touch_marker(MARKER_DOWNLOAD_DONE)
        print("✅ Phase 1 complete.")
    except Exception as e:
        print(f"❌ Phase 1 failed: {e}")
        traceback.print_exc()
        raise

# ══════════════════════════════════════════════════════════
# PHASE 2: Extract archives & convert audio to WAV
# ══════════════════════════════════════════════════════════
if marker_exists(MARKER_CONVERT_DONE):
    print("⏭️  Phase 2 (Convert): Already completed — skipping.")
else:
    print("\n🔄 Phase 2: Extracting archives & converting audio to WAV...")

    import soundfile as sf
    import numpy as np

    def convert_audio_to_wav(src_path, dst_path, target_sr=SAMPLE_RATE):
        """Convert any audio format to mono WAV at target sample rate.
        Uses librosa for robust format support (mp3, flac, ogg, m4a, etc.).
        Falls back to ffmpeg + pydub for edge cases.
        """
        try:
            import librosa
            wav, _ = librosa.load(src_path, sr=target_sr, mono=True)
            sf.write(dst_path, wav, target_sr, subtype='PCM_16')
            return True
        except Exception:
            pass
        # Fallback: ffmpeg via pydub
        try:
            from pydub import AudioSegment
            audio = AudioSegment.from_file(src_path)
            audio = audio.set_frame_rate(target_sr).set_channels(1).set_sample_width(2)
            audio.export(dst_path, format="wav")
            return True
        except Exception:
            pass
        # Fallback: raw ffmpeg
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-i", str(src_path),
                 "-ar", str(target_sr), "-ac", "1", "-sample_fmt", "s16",
                 str(dst_path)],
                check=True, capture_output=True
            )
            return True
        except Exception as e:
            print(f"      ⚠️  Failed to convert {src_path}: {e}")
            return False

    def extract_all_archives(directory):
        """Recursively extract all zip/tar/gz archives found."""
        import zipfile, tarfile
        for root, dirs, files in os.walk(directory):
            for f in files:
                fpath = os.path.join(root, f)
                try:
                    if f.endswith('.zip'):
                        with zipfile.ZipFile(fpath, 'r') as zf:
                            zf.extractall(root)
                        print(f"      📦 Extracted ZIP: {f}")
                    elif f.endswith(('.tar.gz', '.tgz', '.tar.bz2', '.tar')):
                        with tarfile.open(fpath) as tf:
                            tf.extractall(root)
                        print(f"      📦 Extracted TAR: {f}")
                    elif f.endswith('.gz') and not f.endswith('.tar.gz'):
                        import gzip
                        out_path = fpath[:-3]
                        with gzip.open(fpath, 'rb') as gz, open(out_path, 'wb') as out:
                            shutil.copyfileobj(gz, out)
                        print(f"      📦 Extracted GZ: {f}")
                except Exception as e:
                    print(f"      ⚠️  Failed to extract {f}: {e}")

    # Step 2a: If loaded via datasets lib, export audio to disk
    if '_ds_loaded_via_lib' in dir() and _ds_loaded_via_lib:
        print("   Exporting audio from datasets library to WAV files...")
        converted = 0
        failed = 0
        metadata_rows = []
        for split_name in ds:
            split = ds[split_name]
            # Detect audio column
            audio_col = None
            for col in split.column_names:
                if col in ('audio', 'speech', 'wav', 'sound', 'recording'):
                    audio_col = col
                    break
            # Detect text column
            text_col = None
            for col in split.column_names:
                if col in ('text', 'sentence', 'transcription', 'transcript', 'utterance', 'label'):
                    text_col = col
                    break
            if audio_col is None:
                print(f"      ⚠️  No audio column found in split '{split_name}'. Columns: {split.column_names}")
                continue
            if text_col is None:
                print(f"      ⚠️  No text column found in split '{split_name}'. Columns: {split.column_names}")
                continue

            print(f"   Processing split '{split_name}': {len(split)} samples (audio='{audio_col}', text='{text_col}')")
            for idx, sample in enumerate(split):
                try:
                    audio_data = sample[audio_col]
                    text = sample[text_col]
                    if not text or not text.strip():
                        continue
                    file_id = f"{split_name}_{idx:06d}"
                    wav_path = os.path.join(DRIVE_CONVERTED_WAV, f"{file_id}.wav")

                    if os.path.exists(wav_path):
                        metadata_rows.append((file_id, text.strip(), split_name))
                        converted += 1
                        continue

                    if isinstance(audio_data, dict):
                        arr = np.array(audio_data['array'], dtype=np.float32)
                        sr = audio_data['sampling_rate']
                        if sr != SAMPLE_RATE:
                            import librosa
                            arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
                        sf.write(wav_path, arr, SAMPLE_RATE, subtype='PCM_16')
                    elif isinstance(audio_data, str) and os.path.isfile(audio_data):
                        convert_audio_to_wav(audio_data, wav_path)
                    elif isinstance(audio_data, bytes):
                        import io
                        tmp_path = f"/tmp/_audio_tmp_{idx}"
                        with open(tmp_path, 'wb') as f:
                            f.write(audio_data)
                        convert_audio_to_wav(tmp_path, wav_path)
                        os.remove(tmp_path)
                    else:
                        print(f"      ⚠️  Unknown audio format at index {idx}: {type(audio_data)}")
                        failed += 1
                        continue

                    metadata_rows.append((file_id, text.strip(), split_name))
                    converted += 1
                    if converted % 500 == 0:
                        print(f"      ... {converted} files converted")
                except Exception as e:
                    failed += 1
                    if failed <= 5:
                        print(f"      ⚠️  Error at index {idx}: {e}")

        print(f"   Converted: {converted} | Failed: {failed}")
        # Save raw metadata
        meta_path = os.path.join(DRIVE_METADATA, "_all_metadata.json")
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(metadata_rows, f, ensure_ascii=False)
        # Cleanup datasets object from memory
        del ds

    else:
        # Step 2b: Handle snapshot download — extract archives, find audio files
        snapshot_dir = os.path.join(DRIVE_RAW_DATASET, "_hf_snapshot")
        if os.path.isdir(snapshot_dir):
            print("   Extracting any archives in snapshot...")
            extract_all_archives(snapshot_dir)

            # Find all audio files recursively
            AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.wma', '.aac', '.opus'}
            audio_files = []
            for root, dirs, files in os.walk(snapshot_dir):
                for f in files:
                    if Path(f).suffix.lower() in AUDIO_EXTENSIONS:
                        audio_files.append(os.path.join(root, f))
            print(f"   Found {len(audio_files)} audio files.")

            # Convert each to WAV
            converted = 0
            failed = 0
            for af in audio_files:
                stem = Path(af).stem
                wav_dst = os.path.join(DRIVE_CONVERTED_WAV, f"{stem}.wav")
                if os.path.exists(wav_dst):
                    converted += 1
                    continue
                if Path(af).suffix.lower() == '.wav':
                    # Still re-encode to ensure correct sample rate
                    if convert_audio_to_wav(af, wav_dst):
                        converted += 1
                    else:
                        failed += 1
                else:
                    if convert_audio_to_wav(af, wav_dst):
                        converted += 1
                    else:
                        failed += 1
                if converted % 200 == 0:
                    print(f"      ... {converted}/{len(audio_files)} converted")
            print(f"   Converted: {converted} | Failed: {failed}")

            # Try to find metadata files (csv, tsv, jsonl, txt)
            metadata_rows = []
            meta_files = glob.glob(os.path.join(snapshot_dir, '**', '*.csv'), recursive=True)
            meta_files += glob.glob(os.path.join(snapshot_dir, '**', '*.tsv'), recursive=True)
            meta_files += glob.glob(os.path.join(snapshot_dir, '**', '*.jsonl'), recursive=True)
            meta_files += glob.glob(os.path.join(snapshot_dir, '**', 'metadata.txt'), recursive=True)

            import csv
            for mf in meta_files:
                ext = Path(mf).suffix.lower()
                try:
                    if ext == '.jsonl':
                        with open(mf, 'r', encoding='utf-8') as f:
                            for line in f:
                                row = json.loads(line.strip())
                                # Try common field names
                                fid = row.get('file_name', row.get('file', row.get('id', '')))
                                text = row.get('text', row.get('sentence', row.get('transcription', '')))
                                if fid and text:
                                    fid = Path(fid).stem
                                    metadata_rows.append((fid, text.strip(), 'train'))
                    elif ext in ('.csv', '.tsv'):
                        delim = '\t' if ext == '.tsv' else None  # auto-detect
                        with open(mf, 'r', encoding='utf-8') as f:
                            # Try pipe-delimited first (LJSpeech format)
                            first_line = f.readline()
                            f.seek(0)
                            if '|' in first_line:
                                delim = '|'
                            elif '\t' in first_line:
                                delim = '\t'
                            else:
                                delim = ','
                            reader = csv.reader(f, delimiter=delim)
                            for row in reader:
                                if len(row) >= 2:
                                    fid = Path(row[0].strip()).stem
                                    text = row[-1].strip()  # text is usually last column
                                    if fid and text:
                                        metadata_rows.append((fid, text, 'train'))
                    elif ext == '.txt':
                        with open(mf, 'r', encoding='utf-8') as f:
                            for line in f:
                                parts = line.strip().split('|', 1)
                                if len(parts) == 2:
                                    fid = Path(parts[0].strip()).stem
                                    metadata_rows.append((fid, parts[1].strip(), 'train'))
                except Exception as e:
                    print(f"      ⚠️  Error parsing {mf}: {e}")

            if metadata_rows:
                meta_path = os.path.join(DRIVE_METADATA, "_all_metadata.json")
                with open(meta_path, 'w', encoding='utf-8') as f:
                    json.dump(metadata_rows, f, ensure_ascii=False)
                print(f"   Found {len(metadata_rows)} metadata entries.")
            else:
                print("   ⚠️  No metadata found. Will auto-generate from filenames.")

    touch_marker(MARKER_CONVERT_DONE)
    print("✅ Phase 2 complete.")

## 📝 Cell 5 — Generate Model-Compatible Metadata (train.csv / val.csv)

Creates the pipe-delimited metadata files expected by the model's `preprocess_dataset` tool, with a 95/5 train/val split.

In [ ]:
#@title 📝 Generate Metadata (train.csv / val.csv) {display-mode: "form"}

import os, json, random
from pathlib import Path

if marker_exists(MARKER_METADATA_DONE):
    print("⏭️  Phase 3 (Metadata): Already completed — skipping.")
else:
    print("📝 Phase 3: Generating model-compatible metadata files...")

    # Load metadata
    meta_path = os.path.join(DRIVE_METADATA, "_all_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path, 'r', encoding='utf-8') as f:
            all_rows = json.load(f)
    else:
        all_rows = []

    # Get list of actually available WAV files
    available_wavs = {Path(f).stem for f in os.listdir(DRIVE_CONVERTED_WAV) if f.endswith('.wav')}
    print(f"   Available WAV files: {len(available_wavs)}")

    if all_rows:
        # Filter to only entries with actual WAV files
        valid_entries = [(fid, text) for fid, text, *_ in all_rows if fid in available_wavs]
    else:
        # No metadata found — create dummy entries (file_id only, no text)
        # The user will need to provide transcriptions
        print("   ⚠️  No transcription metadata found.")
        print("   Creating entries from WAV filenames (text=filename).")
        print("   ⚠️  You MUST replace the text column with real transcriptions!")
        valid_entries = [(stem, stem) for stem in sorted(available_wavs)]

    print(f"   Valid entries with audio: {len(valid_entries)}")

    if len(valid_entries) == 0:
        raise RuntimeError(
            "No valid entries found. Check that your dataset has both audio files "
            "and matching metadata/transcriptions."
        )

    # Shuffle and split 95/5
    random.seed(42)
    random.shuffle(valid_entries)
    val_size = max(1, int(len(valid_entries) * 0.05))
    val_entries = valid_entries[:val_size]
    train_entries = valid_entries[val_size:]

    print(f"   Train: {len(train_entries)} | Val: {len(val_entries)}")

    # Write pipe-delimited CSVs (file_id|text)
    for filename, entries in [("train.csv", train_entries), ("val.csv", val_entries)]:
        fpath = os.path.join(DRIVE_RAW_DATASET, filename)
        with open(fpath, 'w', encoding='utf-8', newline='\n') as f:
            for fid, text in entries:
                # Clean text: remove pipe chars to avoid breaking CSV format
                text = text.replace('|', ' ').strip()
                f.write(f"{fid}|{text}\n")
        print(f"   Wrote: {fpath}")

    touch_marker(MARKER_METADATA_DONE)
    print("✅ Phase 3 complete.")

## 🧬 Cell 6 — Preprocess Dataset (Phonemes, Mel, Pitch, Energy)

Runs the model's preprocessing pipeline to generate `.npz` (mel/pitch/energy) and `.json` (phoneme IDs) files. Results are saved to Google Drive and symlinked locally.

In [ ]:
#@title 🧬 Preprocess Dataset {display-mode: "form"}

import os, sys, subprocess, shutil
from pathlib import Path

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO
if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)

if marker_exists(MARKER_PREPROCESS_DONE):
    print("⏭️  Phase 4 (Preprocess): Already completed — skipping.")
    print("   Linking preprocessed data to local repo...")
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    if not os.path.exists(LOCAL_DATA_DIR):
        os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)
    print(f"   ✅ Linked: {LOCAL_DATA_DIR} → {DRIVE_PREPROCESSED}")
else:
    print("🧬 Phase 4: Running model preprocessing pipeline...")
    print(f"   Experiment: {EXPERIMENT_NAME}")
    print(f"   Input:  {DRIVE_RAW_DATASET}")
    print(f"   Output: {DRIVE_PREPROCESSED}")

    # If previous partial run exists, remove it
    if os.path.isdir(DRIVE_PREPROCESSED) and os.listdir(DRIVE_PREPROCESSED):
        data_subdir = os.path.join(DRIVE_PREPROCESSED, "data")
        if os.path.isdir(data_subdir):
            existing_npz = len([f for f in os.listdir(data_subdir) if f.endswith('.npz')])
            print(f"   Found {existing_npz} existing preprocessed files from partial run.")
            if existing_npz > 0:
                print("   ⚠️  Removing partial preprocessing output to restart cleanly.")
        shutil.rmtree(DRIVE_PREPROCESSED)
        os.makedirs(DRIVE_PREPROCESSED, exist_ok=True)

    # Run preprocessing
    cmd = [
        sys.executable, "-m", "optispeech.tools.preprocess_dataset",
        EXPERIMENT_NAME,
        DRIVE_RAW_DATASET,
        DRIVE_PREPROCESSED,
        "--n-workers", str(PREPROCESS_WORKERS),
        "--batch-size", "4",
    ]
    print(f"   Command: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=LOCAL_REPO)

    if result.returncode != 0:
        print(f"\n❌ Preprocessing failed with exit code {result.returncode}")
        print("   Check the error messages above for details.")
        raise RuntimeError("Preprocessing failed")

    # Verify output
    data_subdir = os.path.join(DRIVE_PREPROCESSED, "data")
    if os.path.isdir(data_subdir):
        npz_count = len([f for f in os.listdir(data_subdir) if f.endswith('.npz')])
        json_count = len([f for f in os.listdir(data_subdir) if f.endswith('.json')])
        print(f"\n   Generated: {npz_count} .npz files, {json_count} .json files")
    else:
        print("\n   ⚠️  No data subdirectory found in output.")

    # Link to local repo
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    if os.path.exists(LOCAL_DATA_DIR):
        if os.path.islink(LOCAL_DATA_DIR):
            os.unlink(LOCAL_DATA_DIR)
        else:
            shutil.rmtree(LOCAL_DATA_DIR)
    os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)
    print(f"   ✅ Linked: {LOCAL_DATA_DIR} → {DRIVE_PREPROCESSED}")

    touch_marker(MARKER_PREPROCESS_DONE)
    print("\n✅ Phase 4 complete. Phonemes and features saved to Drive.")

## 📊 Cell 7 — Generate Data Statistics

Computes pitch/energy/mel normalization statistics from the preprocessed data. These are required for training and are saved to Drive.

In [ ]:
#@title 📊 Generate Data Statistics {display-mode: "form"}

import os, sys, subprocess, json

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO

if marker_exists(MARKER_STATS_DONE):
    print("⏭️  Phase 5 (Stats): Already completed — skipping.")
    stats_file = os.path.join(DRIVE_DATA_STATS, "stats.json")
    if os.path.exists(stats_file):
        with open(stats_file, 'r') as f:
            stats = json.load(f)
        print("   Saved statistics:")
        for k, v in stats.items():
            print(f"     {k}: {v}")
else:
    print("📊 Phase 5: Computing data statistics...")

    cmd = [
        sys.executable, "-m", "optispeech.tools.generate_data_statistics",
        EXPERIMENT_NAME,
        "-b", "16",
        "-w", "2",
        "-o", DRIVE_DATA_STATS,
    ]
    print(f"   Command: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=LOCAL_REPO)

    if result.returncode != 0:
        print(f"\n❌ Statistics generation failed with exit code {result.returncode}")
        raise RuntimeError("Statistics generation failed")

    stats_file = os.path.join(DRIVE_DATA_STATS, "stats.json")
    if os.path.exists(stats_file):
        with open(stats_file, 'r') as f:
            stats = json.load(f)
        print("\n   Computed statistics:")
        for k, v in stats.items():
            print(f"     {k}: {v}")

        # Auto-update the experiment data config with new statistics
        data_config_path = os.path.join(LOCAL_REPO, "configs", "data", f"{EXPERIMENT_NAME}.yaml")
        if os.path.exists(data_config_path):
            import re
            with open(data_config_path, 'r') as f:
                config_text = f.read()
            for key, value in stats.items():
                pattern = rf'({key}:\s*)([\d.\-]+)'
                config_text = re.sub(pattern, rf'\g<1>{value}', config_text)
            with open(data_config_path, 'w') as f:
                f.write(config_text)
            print(f"\n   ✅ Updated config: {data_config_path}")
    else:
        print("   ⚠️  stats.json not found in output directory.")

    touch_marker(MARKER_STATS_DONE)
    print("\n✅ Phase 5 complete.")

## 🚀 Cell 8 — Train the Model

Launches training with:
- Checkpoints saved directly to Google Drive
- TensorBoard logs on Drive
- **Automatic resume** from the latest checkpoint if training was interrupted
- Mixed precision (16-bit) for speed on Colab GPUs

In [ ]:
#@title 🚀 Train Model {display-mode: "form"}

import os, sys, subprocess, glob
from pathlib import Path

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO
if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)

# Ensure data symlink exists
if not os.path.exists(LOCAL_DATA_DIR):
    os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)
    os.symlink(DRIVE_PREPROCESSED, LOCAL_DATA_DIR)

# ── Find latest checkpoint for resume ──
ckpt_path = None
ckpt_files = sorted(
    glob.glob(os.path.join(DRIVE_CHECKPOINTS, "**", "*.ckpt"), recursive=True),
    key=os.path.getmtime
)
if ckpt_files:
    ckpt_path = ckpt_files[-1]
    print(f"🔄 Resuming from checkpoint: {ckpt_path}")
    print(f"   Modified: {os.path.getmtime(ckpt_path):.0f}")
else:
    print("🆕 Starting fresh training (no checkpoint found).")

# ── Build training command ──
train_cmd = [
    sys.executable, "-m", "optispeech.train",
    f"experiment={EXPERIMENT_NAME}",
    f"trainer=gpu",
    f"trainer.max_steps={MAX_STEPS}",
    f"trainer.precision=16-mixed",
    f"trainer.log_every_n_steps=10",
    f"data.batch_size={BATCH_SIZE}",
    f"data.num_workers=2",
    # Save checkpoints to Drive
    f"callbacks.model_checkpoint.dirpath={DRIVE_CHECKPOINTS}",
    f"callbacks.model_checkpoint.every_n_epochs=1",
    f"callbacks.model_checkpoint.save_top_k=5",
    f"callbacks.model_checkpoint.save_last=true",
    # Logs to Drive
    f"paths.log_dir={DRIVE_LOGS}",
    # Hydra output to Drive
    f"hydra.run.dir={DRIVE_LOGS}/train/{EXPERIMENT_NAME}/runs/${{now:%Y-%m-%d_%H-%M-%S}}",
]

# Add checkpoint resume
if ckpt_path:
    train_cmd.append(f"ckpt_path={ckpt_path}")

print(f"\n🏋️ Training command:")
print(f"   {' '.join(train_cmd)}")
print(f"\n{'='*60}")
print(f"   Experiment:    {EXPERIMENT_NAME}")
print(f"   Batch size:    {BATCH_SIZE}")
print(f"   Max steps:     {MAX_STEPS}")
print(f"   Precision:     16-mixed")
print(f"   Checkpoints:   {DRIVE_CHECKPOINTS}")
print(f"   Logs:          {DRIVE_LOGS}")
if ckpt_path:
    print(f"   Resume from:   {Path(ckpt_path).name}")
print(f"{'='*60}\n")

# ── Launch training ──
result = subprocess.run(train_cmd, cwd=LOCAL_REPO)

if result.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    print(f"\n⚠️  Training exited with code {result.returncode}")
    print("   If this was a Colab disconnection, simply re-run this cell.")
    print("   Training will resume automatically from the latest checkpoint.")

## 📈 Cell 9 — TensorBoard (Optional)

Monitor training progress in real-time.

In [ ]:
#@title 📈 Launch TensorBoard {display-mode: "form"}

%load_ext tensorboard
%tensorboard --logdir {DRIVE_LOGS}

## 🔍 Cell 10 — Inspect Checkpoints & Project Status

In [ ]:
#@title 🔍 Project Status {display-mode: "form"}

import os, glob, json
from pathlib import Path
from datetime import datetime

print("📋 Project Status Report")
print("=" * 50)

# Phase markers
phases = [
    ("Download",    MARKER_DOWNLOAD_DONE),
    ("Convert",     MARKER_CONVERT_DONE),
    ("Metadata",    MARKER_METADATA_DONE),
    ("Preprocess",  MARKER_PREPROCESS_DONE),
    ("Statistics",  MARKER_STATS_DONE),
]
print("\n🏁 Completed Phases:")
for name, marker in phases:
    status = "✅" if os.path.exists(marker) else "⬜"
    print(f"   {status} {name}")

# WAV files
wav_count = len(glob.glob(os.path.join(DRIVE_CONVERTED_WAV, "*.wav")))
print(f"\n🎵 WAV files on Drive: {wav_count}")

# Preprocessed files
data_dir = os.path.join(DRIVE_PREPROCESSED, "data")
if os.path.isdir(data_dir):
    npz_count = len(glob.glob(os.path.join(data_dir, "*.npz")))
    print(f"🧬 Preprocessed samples: {npz_count}")

# Checkpoints
ckpts = sorted(
    glob.glob(os.path.join(DRIVE_CHECKPOINTS, "**", "*.ckpt"), recursive=True),
    key=os.path.getmtime
)
print(f"\n💾 Checkpoints on Drive: {len(ckpts)}")
for c in ckpts[-5:]:
    size_mb = os.path.getsize(c) / (1024 * 1024)
    mtime = datetime.fromtimestamp(os.path.getmtime(c)).strftime('%Y-%m-%d %H:%M')
    print(f"   📄 {Path(c).name} ({size_mb:.1f} MB) — {mtime}")

# Drive usage
def dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total

total_gb = dir_size(DRIVE_PROJECT_ROOT) / (1024**3)
print(f"\n💽 Total Drive usage: {total_gb:.2f} GB")

## 🔄 Cell 11 — Reset a Phase (Optional)

Use this to re-run a specific phase. Select which phase marker to remove.

In [ ]:
#@title 🔄 Reset Phase Marker (Optional) {display-mode: "form"}

#@markdown Select which phase to reset:
PHASE_TO_RESET = "None"  #@param ["None", "Download", "Convert", "Metadata", "Preprocess", "Statistics", "ALL"]

import os

phase_map = {
    "Download":   MARKER_DOWNLOAD_DONE,
    "Convert":    MARKER_CONVERT_DONE,
    "Metadata":   MARKER_METADATA_DONE,
    "Preprocess": MARKER_PREPROCESS_DONE,
    "Statistics": MARKER_STATS_DONE,
}

if PHASE_TO_RESET == "None":
    print("ℹ️  No phase selected for reset.")
elif PHASE_TO_RESET == "ALL":
    for name, path in phase_map.items():
        if os.path.exists(path):
            os.remove(path)
            print(f"   🗑️ Reset: {name}")
    print("✅ All phases reset. Re-run cells 4–7 to redo.")
else:
    marker = phase_map.get(PHASE_TO_RESET)
    if marker and os.path.exists(marker):
        os.remove(marker)
        print(f"✅ Phase '{PHASE_TO_RESET}' reset. Re-run the corresponding cell.")
    else:
        print(f"ℹ️  Phase '{PHASE_TO_RESET}' was not marked as complete.")